[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/campos_ondas_electromagneticas/blob/main/unidad_01/03_capacitancia_inductancia_energia.ipynb)

Pulse el botón para ejecutar este notebook en Google Colab sin instalar nada.

# Semana 3 — Capacitancia, inductancia y energía

**Campos y Ondas Electromagnéticas (ICEE1033) — Unidad 1**

## 1. Objetivos de aprendizaje

Al terminar este notebook usted podrá:

1. Calcular la capacitancia de un condensador de placas y la energía que
   almacena.
2. Obtener $C'$ y $L'$ de un cable coaxial a partir de su geometría.
3. Reconocer que $\sqrt{L'/C'}$ y $1/\sqrt{L'C'}$ son la impedancia
   característica y la velocidad de la onda.
4. Explicar por qué la energía crece con el **cuadrado** del voltaje.

In [ ]:
# Preparación del entorno: local o Google Colab, con verificación SHA256.
import hashlib
import sys
import urllib.request
from pathlib import Path

MODULOS = {
    "utilidades_notebook.py": "e1811892d086ca99e03694c0e70853886f63be58326e3e7232c6978db1fdf7a9",
    "constantes_fisicas.py": "394c39ad2aac2f4870620e3df6e276046f04abb811d1b28b6fa14f1836fe20ce",
    "campos_electrostaticos.py": "d7ccfcce6a4cb2547f71f1f05bdb8f3912589035677b46ca8547a36cee0d072d",
}
URL_SRC = (
    "https://raw.githubusercontent.com/"
    "gsilvaoelker/campos_ondas_electromagneticas/main/src/"
)


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


candidatos = [Path.cwd(), *Path.cwd().parents]
raiz_repo = next((p for p in candidatos if (p / ".git").exists()), None)
if raiz_repo is not None:
    for modulo, esperado in MODULOS.items():
        archivo = raiz_repo / "src" / modulo
        if not archivo.exists() or sha256(archivo) != esperado:
            raise RuntimeError(
                f"Hash local desactualizado para {modulo}. "
                "Ejecute scripts/refresh_notebook_hashes.py."
            )
    raiz = raiz_repo
else:
    raiz = Path.cwd()
    (raiz / "src").mkdir(exist_ok=True)
    for modulo, esperado in MODULOS.items():
        destino = raiz / "src" / modulo
        if destino.exists() and sha256(destino) == esperado:
            continue
        with urllib.request.urlopen(URL_SRC + modulo, timeout=30) as respuesta:
            datos = respuesta.read()
        obtenido = hashlib.sha256(datos).hexdigest()
        if obtenido != esperado:
            raise RuntimeError(
                f"SHA256 inválido para {modulo}: {obtenido} != {esperado}"
            )
        destino.write_bytes(datos)

sys.path.insert(0, str(raiz / "src"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from constantes_fisicas import VELOCIDAD_LUZ
from campos_electrostaticos import (
    capacitancia_placas_paralelas,
    capacitancia_coaxial,
    inductancia_coaxial,
    energia_electrica,
    energia_magnetica,
)
from utilidades_notebook import configurar_estilo_graficos, tabla_resultados

configurar_estilo_graficos()

## 2. De dónde sale todo esto

### 2.1 La capacitancia solo depende de la forma

La capacitancia dice cuánta carga acepta una estructura por cada volt que se
le aplica. Es una propiedad **geométrica y material**: no depende del voltaje
ni de la carga, solo de la forma de los conductores y del material entre
ellos.

El modelo de placas paralelas supone campo uniforme adentro y cero afuera.
Eso vale mientras la separación sea mucho menor que el tamaño de las placas.

### 2.2 La inductancia y el logaritmo

En un coaxial el campo magnético existe solo entre los dos conductores. Para
obtener el flujo hay que integrar $B_\phi = \mu I/(2\pi\rho)$ desde $a$ hasta
$b$, y de ahí sale el logaritmo.

Note que $C'$ tiene el $\ln(b/a)$ **abajo** y $L'$ lo tiene **arriba**. Por
eso, al multiplicarlas, el logaritmo se cancela y la geometría desaparece.
Ésa es la razón de que la velocidad no dependa de la forma del cable.

### 2.3 El factor 1/2 de la energía

Cargar un condensador desde cero hasta $V$ requiere un trabajo
$\int_0^V C v\,dv = \tfrac{1}{2}CV^2$. El medio aparece porque las primeras
cargas se mueven contra un potencial menor que las últimas. Con la
inductancia el razonamiento es idéntico, cambiando voltaje por corriente.

## 3. Ecuaciones

**Condensador de placas** de área $A$ y separación $d$:

$$
E = \frac{V}{d},
\qquad
C = \frac{\varepsilon_0 \varepsilon_r A}{d},
\qquad
Q = C V .
$$

**Cable coaxial** de radios $a < b$, por unidad de longitud:

$$
C' = \frac{2\pi\varepsilon_0\varepsilon_r}{\ln(b/a)},
\qquad
L' = \frac{\mu_0 \ln(b/a)}{2\pi}.
$$

**Energías:**

$$
W_e = \frac{1}{2} C V^2,
\qquad
W_m = \frac{1}{2} L I^2 .
$$

**Lo que anticipa la Unidad 3:**

$$
Z_0 = \sqrt{\frac{L'}{C'}},
\qquad
u_p = \frac{1}{\sqrt{L'C'}} = \frac{c}{\sqrt{\varepsilon_r \mu_r}} .
$$

La segunda igualdad sale de sustituir $C'$ y $L'$: el $\ln(b/a)$ se cancela y
solo quedan las propiedades del material.

## 4. Qué significa físicamente

**La capacitancia no cambia con el voltaje.** Al duplicar $V$ se duplica $Q$,
así que el cociente $Q/V$ queda igual. Lo que sí cambia, y al cuadrado, es la
energía.

**El dieléctrico permite guardar más carga.** Sus moléculas se polarizan y
crean un campo interno que se opone al aplicado. Resultado: con el mismo
voltaje cabe más carga. El factor de mejora es exactamente $\varepsilon_r$.

**En el coaxial la señal viaja más lento que la luz.** Con
$\varepsilon_r = 2.30$ resulta $u_p \approx 0.66\,c$. Por eso una señal por
cable llega después que la misma señal por el aire.

**La geometría fija $Z_0$, el material fija $u_p$.** Cambiar la relación
$b/a$ modifica la impedancia característica —así se fabrican los cables de
50 y de 75 $\Omega$— pero no altera la velocidad.

## 5. Parámetros modificables

Esta es la única celda que conviene editar. Cambie un valor, ejecute el notebook
completo y compare con lo que tenía antes.

In [ ]:
# --- Problema 1: condensador de placas ---
area_placas = 0.0100        # área de cada placa [m^2]
separacion_placas = 0.002   # separación entre placas [m]
eps_r_placas = 2.20         # permitividad relativa del dieléctrico
voltaje_placas = 12.0       # voltaje aplicado [V]

# --- Problema 2: cable coaxial ---
radio_interno = 0.001       # radio del conductor interior a [m]
radio_externo = 0.004       # radio del conductor exterior b [m]
longitud_linea = 0.500      # longitud del cable [m]
eps_r_coaxial = 2.30        # permitividad relativa del aislante
voltaje_coaxial = 5.00      # voltaje entre conductores [V]
corriente_coaxial = 0.200   # corriente que circula [A]

## 6. Implementación

### 6.1 Problema 1 — placas paralelas

In [ ]:
campo_placas = voltaje_placas / separacion_placas
C_placas = capacitancia_placas_paralelas(
    area_placas, separacion_placas, eps_r_placas
)
carga_placas = C_placas * voltaje_placas
energia_placas = energia_electrica(C_placas, voltaje_placas)

### 6.2 Problema 2 — el cable coaxial

In [ ]:
C_por_metro = capacitancia_coaxial(radio_interno, radio_externo, eps_r_coaxial)
L_por_metro = inductancia_coaxial(radio_interno, radio_externo)

C_total = C_por_metro * longitud_linea
L_total = L_por_metro * longitud_linea

energia_electrica_coaxial = energia_electrica(C_total, voltaje_coaxial)
energia_magnetica_coaxial = energia_magnetica(L_total, corriente_coaxial)

impedancia_caracteristica = np.sqrt(L_por_metro / C_por_metro)
velocidad_fase = 1.0 / np.sqrt(L_por_metro * C_por_metro)

## 7. Resultados numéricos

In [ ]:
tabla_resultados(
    [
        ("Campo entre las placas", "E", campo_placas, "V/m"),
        ("Capacitancia", "C", C_placas, "F"),
        ("Carga almacenada", "Q", carga_placas, "C"),
        ("Energía almacenada", "W_e", energia_placas, "J"),
    ]
)

In [ ]:
tabla_resultados(
    [
        ("Capacitancia por metro", "C'", C_por_metro, "F/m"),
        ("Inductancia por metro", "L'", L_por_metro, "H/m"),
        ("Energía eléctrica", "W_e", energia_electrica_coaxial, "J"),
        ("Energía magnética", "W_m", energia_magnetica_coaxial, "J"),
        ("Impedancia característica", "Z_0", impedancia_caracteristica, "ohm"),
        ("Velocidad de la onda", "u_p", velocidad_fase, "m/s"),
    ]
)

Comprobación: la velocidad debe dar $c/\sqrt{\varepsilon_r}$, sin ninguna
referencia a la geometría.

In [ ]:
velocidad_predicha = VELOCIDAD_LUZ / np.sqrt(eps_r_coaxial)
print(f"1 / sqrt(L' C')      = {velocidad_fase:.6e} m/s")
print(f"c / sqrt(eps_r)      = {velocidad_predicha:.6e} m/s")
print(f"Error relativo       = {abs(velocidad_fase / velocidad_predicha - 1.0):.3e}")
print(f"Fracción de c        = {velocidad_fase / VELOCIDAD_LUZ:.4f}")

## 8. Visualización

Energía almacenada en función del voltaje. El punto marca el caso de trabajo.

In [ ]:
voltajes = np.linspace(0.0, 2.0 * voltaje_placas, 200)
energias = energia_electrica(C_placas, voltajes)

fig, eje = plt.subplots()
eje.plot(voltajes, energias * 1.0e9)
eje.scatter([voltaje_placas], [energia_placas * 1.0e9], color="black", zorder=5,
            label="caso de trabajo")
eje.set_xlabel("Voltaje (V)")
eje.set_ylabel("Energía eléctrica (nJ)")
eje.set_title("La energía crece con el cuadrado del voltaje")
eje.legend()
fig.tight_layout()
plt.show()

## 9. Qué nos dicen los resultados

**La curva es una parábola, no una recta.** Duplicar el voltaje **cuadruplica**
la energía. Por eso el aislamiento de un condensador es crítico: la energía
que se libera en una falla crece mucho más rápido que la tensión que la causa.

**En este caso $W_m$ supera a $W_e$.** No es una regla general: depende de los
valores de $V$ e $I$ elegidos. Las dos energías se igualan cuando
$V/I = Z_0$, es decir, cuando la línea está terminada en su impedancia
característica. Ese equilibrio es lo que hace especial a $Z_0$.

**$Z_0$ dio cerca de 55 $\Omega$.** Muy cerca de los 50 $\Omega$ de un cable
comercial. Se ajusta cambiando la relación $b/a$.

**La velocidad coincide con $c/\sqrt{\varepsilon_r}$.** La comprobación da un
error relativo del orden de $10^{-16}$: la geometría se canceló, como
anticipaba la sección 2.

**Esto no es casualidad.** Que dos parámetros de una estructura estática
determinen la impedancia y la velocidad de una onda es justamente el modelo de
línea de transmisión de la Unidad 3.

## 10. Ejercicios para experimentar

            1. Duplique `separacion_placas`. ¿Qué pasa con $C$, $Q$ y $W_e$ a voltaje
               constante? ¿Y con el campo $E$?
            2. Cambie `eps_r_placas` a `1.0` (vacío) y luego a `80.0` (agua). ¿Cuánta
               energía más guarda en el segundo caso?
            3. Ajuste `radio_externo` hasta obtener $Z_0 \approx 50~\Omega$. ¿Qué relación
               $b/a$ le queda?
            4. Cambie `eps_r_coaxial` a `1.0`. ¿Cuánto vale ahora $u_p$? ¿Y $Z_0$? ¿Cuál
               de los dos cambió?
            5. Ajuste `corriente_coaxial` hasta que $W_e = W_m$. Compare
               `voltaje_coaxial / corriente_coaxial` con $Z_0$.
            6. Duplique `longitud_linea`. ¿Cambian $C'$ y $L'$? ¿Cambian $W_e$ y $W_m$?
               ¿Cambia $Z_0$? Justifique cada una.